# `gold.bridge_ticker_manager` — DDL

Resolves the many-to-many between trusts and managers. It is genuinely many-to-many in both
directions, which is what earns the table: 70 of 97 trusts have more than one named manager,
and three managers run two trusts each — Sat Duhra (BNKR, HFEL), Simon Gergel (BUT, MRCH)
and Anthony Lynch (JCH, MRC). Without those three it would be one-to-many and no bridge
would be needed.

Keyed on the **versioned** `ticker_key`, so a manager change opens a new ticker version and
a new set of bridge rows.

`allocation_factor` is how a manager-level count is kept honest. Ignore it and a trust
counts once per manager, which is right for "what share of managers beat the index". Apply
it and the weights sum to one per trust, which is right for anything that must reconcile to
the 440 rows in the fact.

Declared here rather than inside the load, so the shape can be read without reading the
code that fills it. Catalog name has hyphens, so every reference needs backticks.

In [ ]:
CREATE TABLE IF NOT EXISTS `index-vs-trust-pipeline`.gold.bridge_ticker_manager (
  ticker_key        STRING COMMENT 'The versioned dim_ticker key, so a manager change reopens the bridge',
  manager_key       STRING COMMENT 'Joins dim_manager',
  manager_position  INT    COMMENT '1 is the first name listed. Source order, which is not seniority',
  allocation_factor DOUBLE COMMENT 'One over the manager count. Weights sum to 1 per trust'
)
COMMENT 'Trust to manager, many to many. 230 rows over 97 trusts and 227 managers';

## Verification

Expected: the table exists, with the column count stated in
`specs/10_manager_dimensions/manager-dimensions.md`.

In [ ]:
SELECT COUNT(*) AS columns
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'gold' AND table_name = 'bridge_ticker_manager';